# C06 — Detecção de Antinomias (Conflitos Candidatos entre Normas)

**Disciplina:** Sistemas Cognitivos com Large Language Models (INFNET, 26E2_3)

**Autor:** Anderson Corrêa

**Projeto:** Letra da Lei

Este notebook vai além dos cinco pontos exigidos pelo projeto: explora um detector de
antinomias (conflitos normativos) sobre o corpus, uma análise adicional construída sobre
os mesmos módulos de recuperação e o modelo local.

O pipeline, em três etapas:

1. **Geração de candidatos** — recuperação por similaridade
   restringe o espaço de pares de dispositivos
2. **Adjudicação** — para cada candidato, um princípio de resolução
   LINDB é calculado e servido ao modelo local, que julga se há conflito plausível
3. **Avaliação** — precisão/revocação/F1 contra um gold-set
   pequeno e **ilustrativo**, com a limitação de não ter sido verificado por especialista

## Setup

In [1]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import time
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "direito_dados").exists():
    _repo_root = Path(__file__).resolve().parent if "__file__" in globals() else _repo_root
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from direito_dados.corpus import load_corpus, NORMS
from direito_dados.corpus.models import HierarchyLevel
from direito_dados.retrieval.chunks import chunk_corpus
from direito_dados.retrieval.embedder import E5Embedder
from direito_dados.retrieval.index import VectorIndex
from direito_dados.generation.llm import OllamaClient, ollama_available, ollama_has_model
from direito_dados.conflicts.candidates import generate_candidates
from direito_dados.conflicts.detect import detect_conflicts
from direito_dados.conflicts.principles import deterministic_principle, principle_hint, ResolutionPrinciple
from direito_dados.conflicts.evaluate import GoldAntinomy, evaluate_antinomias

MODEL = "llama3.1:8b"
assert ollama_available(), "Ollama precisa estar rodando em localhost:11434"
assert ollama_has_model(MODEL), f"Modelo {MODEL} precisa estar baixado (`ollama pull {MODEL}`)"
llm = OllamaClient(model=MODEL)

t0 = time.time()
corpus = load_corpus("data/raw", specs=[NORMS["CP"]])
chunks = chunk_corpus(corpus)
embedder = E5Embedder()
index = VectorIndex.build(chunks, embedder)
chunks_by_id = {c.id: c for c in chunks}
print(f"Índice construído em {time.time() - t0:.1f}s | {len(chunks)} dispositivos do CP indexados")

Índice construído em 16.2s | 431 dispositivos do CP indexados


## 1. Geração de candidatos

Para cada dispositivo em vigor, o gerador de candidatos consulta seus `k` vizinhos
semânticos mais próximos (excluindo revogados) e mantém os pares cuja similaridade máxima
ultrapassa um limiar (`threshold`). Usamos os mesmos parâmetros do teste de integração do
projeto (`k=3`, `threshold=0.85`).

In [2]:
t0 = time.time()
candidates = generate_candidates(chunks, index, embedder, k=3, threshold=0.85)
print(f"Candidatos gerados em {time.time() - t0:.1f}s: {len(candidates)} pares acima do limiar")
print("\nTop 15 por similaridade:")
for cp in candidates[:15]:
    print(f"  {cp.a:<16} × {cp.b:<16} sim={cp.similarity:.3f}")

Candidatos gerados em 29.9s: 813 pares acima do limiar

Top 15 por similaridade:
  CP:art210        × CP:art211        sim=0.906
  CP:art125        × CP:art126        sim=0.901
  CP:art197        × CP:art199        sim=0.899
  CP:art124        × CP:art126        sim=0.899
  CP:art198        × CP:art199        sim=0.898
  CP:art296        × CP:art306        sim=0.897
  CP:art359-O      × CP:art359-Q      sim=0.897
  CP:art197        × CP:art198        sim=0.897
  CP:art297        × CP:art298        sim=0.896
  CP:art252        × CP:art253        sim=0.895
  CP:art213        × CP:art217-A      sim=0.894
  CP:art359-O      × CP:art359-U      sim=0.893
  CP:art270        × CP:art271        sim=0.893
  CP:art43         × CP:art45         sim=0.893
  CP:art297        × CP:art299        sim=0.892


**Leitura:** o índice contém 431 dispositivos do CP (célula anterior); a maior parte está
em vigor, com uma minoria de dispositivos revogados mantidos no índice para fins de
citação/consulta, mas excluídos da geração de candidatos. Mesmo restringindo aos vigentes,
a comparação exaustiva de todos os pares seria da ordem de dezenas de milhares; a restrição
por similaridade reduz isso a **813 candidatos** acima do limiar de 0.85 — ainda muitos para
adjudicar todos com um LLM local. Adjudicar exaustivamente os 813 pares é possível em
princípio, mas fica fora do orçamento deste notebook: por isso a próxima etapa roda sobre um
subconjunto limitado (os 12 pares de maior similaridade), uma escolha de orçamento de
execução para manter as chamadas ao Ollama dentro de um tempo razoável neste notebook.

## 2. Princípios LINDB determinísticos vs. adjudicação por LLM

A **LINDB** (Lei de Introdução às Normas do Direito Brasileiro) reúne as regras que dizem
qual norma prevalece quando duas normas conflitam. Três critérios guiam essa escolha:
*lex superior* — a norma de hierarquia superior prevalece sobre a inferior; *lex posterior*
— a lei mais nova prevalece sobre a mais antiga; e *lex specialis* — a lei específica
prevalece sobre a geral.

Antes de adjudicar, vale explicitar a divisão de trabalho: *lex superior* (hierarquia) e
*lex posterior* (data) são **computáveis deterministicamente** a partir de metadados. *Lex specialis*
(especificidade — qual norma trata do caso de forma mais específica) exige **ler e
comparar o conteúdo** dos dois dispositivos, então fica a cargo do modelo.

Como todos os 431 dispositivos aqui vêm do mesmo Código Penal (mesma hierarquia, mesma
data de origem), a dica gerada automaticamente para o modelo sempre recomenda "avaliar lex
specialis" para os pares do Código Penal. O caso determinístico só aparece comparando
normas de hierarquias/datas diferentes. Demonstramos os três ramos com exemplos sintéticos
de metadados (sem chamar o LLM, já que a lógica é 100% determinística):

In [3]:
examples = [
    ("CF vs. CP (hierarquias diferentes)", HierarchyLevel.CONSTITUICAO, HierarchyLevel.DECRETO_LEI, 1988, 1940),
    ("Duas leis ordinárias, anos diferentes", HierarchyLevel.LEI_ORDINARIA, HierarchyLevel.LEI_ORDINARIA, 1990, 2006),
    ("Dois artigos do mesmo CP (caso real deste notebook)", HierarchyLevel.DECRETO_LEI, HierarchyLevel.DECRETO_LEI, None, None),
]
for label, level_a, level_b, year_a, year_b in examples:
    principle = deterministic_principle(level_a, level_b, year_a, year_b)
    hint = principle_hint(level_a, level_b, year_a, year_b)
    print(f"{label}:\n  princípio determinístico = {principle.value}\n  dica ao LLM = {hint!r}\n")

CF vs. CP (hierarquias diferentes):
  princípio determinístico = lex_superior
  dica ao LLM = 'Hierarquia distinta: lex superior pode prevalecer (norma superior derroga inferior).'

Duas leis ordinárias, anos diferentes:
  princípio determinístico = lex_posterior
  dica ao LLM = 'Mesma hierarquia, datas distintas: lex posterior pode prevalecer (norma posterior derroga anterior).'

Dois artigos do mesmo CP (caso real deste notebook):
  princípio determinístico = undetermined
  dica ao LLM = 'Mesma hierarquia e data: avaliar lex specialis (a norma mais específica pode prevalecer).'



## 3. Adjudicação: candidatos → conflitos candidatos

Cada par candidato é adjudicado pelo LLM local: o pipeline monta a dica de princípio LINDB,
apresenta os dois textos ao modelo, e mantém apenas os pares em que o modelo julgou haver
conflito plausível com confiança acima do limiar (`min_confidence`). O princípio final
registrado vem do modelo quando ele se posiciona (tipicamente *lex specialis*, aqui); só
cai para o princípio determinístico quando o modelo não se posiciona.

In [4]:
top12 = candidates[:12]
t0 = time.time()
conflicts = detect_conflicts(top12, chunks_by_id, corpus, llm, min_confidence=0.5)
print(f"Adjudicados {len(top12)} pares em {time.time() - t0:.1f}s -> {len(conflicts)} confirmados como conflito candidato\n")
for c in conflicts:
    print(f"{c.a:<16} × {c.b:<16} | princípio={c.principle:<14} confiança={c.confidence:.2f}")
    if c.rationale:
        print(f"    razão: {c.rationale}")

Adjudicados 12 pares em 53.1s -> 9 confirmados como conflito candidato

CP:art210        × CP:art211        | princípio=lex_specialis  confiança=1.00
CP:art125        × CP:art126        | princípio=lex specialis  confiança=0.80
CP:art197        × CP:art199        | princípio=lex_specialis  confiança=1.00
CP:art124        × CP:art126        | princípio=lex_specialis  confiança=0.90
CP:art197        × CP:art198        | princípio=lex_specialis  confiança=0.90
CP:art297        × CP:art298        | princípio=lex_specialis  confiança=0.80
    razão: Os dispositivos A e B possuem a mesma hierarquia (Código Penal) e data de publicação, mas dispositivo A se refere específicamente à falsificação de documentos públicos e particulares relacionados à previdência social, enquanto dispositivo B se refere à falsificação de cartões de crédito ou débito. Portanto, é possível argumentar que a norma mais específica (lex specialis) prevaleça no caso da falsificação de cartões, mas não há uma regra explíci

**Leitura:** dos 12 pares adjudicados, **9** foram confirmados como conflito candidato
nesta execução. Os 3 pares que não passaram pela adjudicação — `CP:art198` × `CP:art199`,
`CP:art296` × `CP:art306` e `CP:art359-O` × `CP:art359-Q` — foram julgados sem conflito
plausível ou com confiança abaixo do limiar (`min_confidence=0.5`) pelo modelo. A taxa de
confirmação alta (75%) ainda é consistente com o padrão esperado: a etapa de geração
(seção 1) já filtrou por altíssima similaridade temática, então a maioria dos pares
realmente trata de matéria sobreposta.

**Importante:** "confirmado" aqui significa apenas que o LLM local julgou plausível uma
antinomia, não que exista de fato uma contradição normativa juridicamente estabelecida.
Nenhum desses pares vira uma aresta de conflito confirmado no grafo — todos permanecem
marcados como candidatos para revisão humana. Mesmo com uma minoria rejeitada, o **limiar
de similaridade da geração de candidatos, não a adjudicação, continua sendo quem faz a
triagem mais forte**.

## 4. Avaliação: gold-set ilustrativo

Para medir precisão/revocação, construímos um gold-set de **3 pares**, cada um justificado
por conhecimento jurídico geral (não por consulta a um especialista em direito penal, sendo essa
a limitação central desta seção):

- **`CP:art213` × `CP:art217-A`** (estupro / estupro de vulnerável): fricção doutrinária
  real e conhecida — quando a vítima é vulnerável *e* há violência ou grave ameaça, a
  aplicação isolada de um ou outro tipo penal, ou o concurso entre eles, é objeto de debate
  na doutrina e na jurisprudência penal brasileira. Um caso canônico de possível
  *lex specialis* (art. 217-A é mais específico quanto ao sujeito passivo).
- **`CP:art197` × `CP:art198`** (atentado contra a liberdade de trabalho / constranger a
  celebrar contrato de trabalho): relação clássica de norma geral vs. norma especial — o
  art. 197 tipifica o constrangimento à liberdade de trabalho de forma genérica, e o art.
  198 tipifica uma manifestação específica dessa mesma conduta (forçar a celebração de um
  contrato de trabalho específico).
- **`CP:art124` × `CP:art126`** (autoaborto/aborto consentido pela gestante / aborto
  provocado por terceiro com consentimento da gestante): a família de crimes de aborto do
  CP distribui a mesma conduta nuclear (interrupção da gravidez) por diferentes graus de
  participação e consentimento — um terreno clássico de sobreposição normativa discutido em
  doutrina de direito penal.

**Este não é um gold-set validado por um especialista em direito penal**: Um gold-set real exigiria revisão por um profissional da área, fora do escopo deste projeto.

In [5]:
gold = [
    GoldAntinomy("CP:art213", "CP:art217-A"),
    GoldAntinomy("CP:art197", "CP:art198"),
    GoldAntinomy("CP:art124", "CP:art126"),
]

metrics = evaluate_antinomias(conflicts, gold)
print(f"precisão:  {metrics['precision']:.3f}  (tp={metrics['tp']}, fp={metrics['fp']})")
print(f"revocação: {metrics['recall']:.3f}  (tp={metrics['tp']}, fn={metrics['fn']})")
print(f"F1:        {metrics['f1']:.3f}")

precisão:  0.333  (tp=3, fp=6)
revocação: 1.000  (tp=3, fn=0)
F1:        0.500


**Leitura:** nesta execução, os três pares do gold-set caíram entre os 12 candidatos de
maior similaridade (seção 1) **e** os três foram confirmados pelo LLM (seção 3) —
revocação (recall) = 1.000 (tp=3, fn=0): nenhum par do gold-set foi perdido, nem na
geração de candidatos, nem na adjudicação. A precisão, no entanto, é baixa (0.333, tp=3,
fp=6): dos 9 conflitos candidatos confirmados, 6 não estão no gold-set de 3 itens
(`CP:art210`×`CP:art211`, `CP:art125`×`CP:art126`, `CP:art197`×`CP:art199`,
`CP:art297`×`CP:art298`, `CP:art252`×`CP:art253`, `CP:art359-O`×`CP:art359-U`). Isso
**não** significa que esses 6 pares sejam falsos positivos no sentido jurídico — são
candidatos igualmente plausíveis que simplesmente não foram incluídos neste gold-set
minúsculo e ilustrativo. Com apenas 3 pares no gold-set contra 9 conflitos candidatos
confirmados, a precisão numérica é baixa.

O recall perfeito **não é uma garantia do pipeline**, mas o resultado desta
execução específica.

Este resultado é ilustrativo do método de avaliação, não uma medida confiável de qualidade
do detector.

## Conclusão

Este notebook demonstrou o detector de antinomias sobre o Código Penal: geração
de candidatos por similaridade semântica, a separação de responsabilidades entre
princípios LINDB determinísticos (*lex superior*/*lex posterior*, computados de metadados,
sem LLM) e a adjudicação por LLM da *lex specialis* — que exige leitura e comparação de
conteúdo, e uma avaliação precisão/revocação/F1 contra um gold-set pequeno e ilustrativo. 

O ponto mais importante: **cada conflito é um candidato para revisão, nunca um veredito**.